In [1]:
from fastai.text.all import *

In [2]:
path = untar_data(URLs.HUMAN_NUMBERS)

In [3]:
path.ls()

(#2) [Path('/home/cgb2/.fastai/data/human_numbers/train.txt'),Path('/home/cgb2/.fastai/data/human_numbers/valid.txt')]

In [4]:
lines = L()
with open(path/'train.txt') as f: lines += L(*f.readlines())
with open(path/'valid.txt') as f: lines += L(*f.readlines())

In [5]:
lines

(#9998) ['one \n','two \n','three \n','four \n','five \n','six \n','seven \n','eight \n','nine \n','ten \n'...]

In [9]:
text= ' . '.join([l.strip() for l in lines])

In [24]:
tokens= text.split(' ')
tokens[:10]

['one', '.', 'two', '.', 'three', '.', 'four', '.', 'five', '.']

In [25]:
vocab = L(*tokens).unique()
vocab

(#30) ['one','.','two','three','four','five','six','seven','eight','nine'...]

In [26]:
word2idx = {w:i for i,w in enumerate(vocab)}
word2idx

{'one': 0,
 '.': 1,
 'two': 2,
 'three': 3,
 'four': 4,
 'five': 5,
 'six': 6,
 'seven': 7,
 'eight': 8,
 'nine': 9,
 'ten': 10,
 'eleven': 11,
 'twelve': 12,
 'thirteen': 13,
 'fourteen': 14,
 'fifteen': 15,
 'sixteen': 16,
 'seventeen': 17,
 'eighteen': 18,
 'nineteen': 19,
 'twenty': 20,
 'thirty': 21,
 'forty': 22,
 'fifty': 23,
 'sixty': 24,
 'seventy': 25,
 'eighty': 26,
 'ninety': 27,
 'hundred': 28,
 'thousand': 29}

In [27]:
nums= L(word2idx[i] for i in tokens)

In [28]:
nums

(#63095) [0,1,2,1,3,1,4,1,5,1...]

In [29]:
L((tokens[i:i+3],tokens[i+3]) for i in range(0,len(tokens)-4,3))

(#21031) [(['one', '.', 'two'], '.'),(['.', 'three', '.'], 'four'),(['four', '.', 'five'], '.'),(['.', 'six', '.'], 'seven'),(['seven', '.', 'eight'], '.'),(['.', 'nine', '.'], 'ten'),(['ten', '.', 'eleven'], '.'),(['.', 'twelve', '.'], 'thirteen'),(['thirteen', '.', 'fourteen'], '.'),(['.', 'fifteen', '.'], 'sixteen')...]

- 이전 세단어를 기반으로 다음단어를 예측하자. 

In [31]:
seqs = L((tensor(nums[i:i+3]), nums[i+3]) for i in range(0,len(nums)-4,3))
seqs

(#21031) [(tensor([0, 1, 2]), 1),(tensor([1, 3, 1]), 4),(tensor([4, 1, 5]), 1),(tensor([1, 6, 1]), 7),(tensor([7, 1, 8]), 1),(tensor([1, 9, 1]), 10),(tensor([10,  1, 11]), 1),(tensor([ 1, 12,  1]), 13),(tensor([13,  1, 14]), 1),(tensor([ 1, 15,  1]), 16)...]

In [34]:
bs = 64 
cut = int(len(seqs)*0.8)
dls = DataLoaders.from_dsets(seqs[:cut],seqs[cut:],bs=64,shuffle=False)

In [36]:
dls

In [38]:
class LMModel1(Module): 
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz,n_hidden)
        self.h_h = nn.Linear(n_hidden,n_hidden)
        self.h_o = nn.Linear(n_hidden,vocab_sz) 
    def forward(self,x):
        h= F.relu(self.h_h(self.i_h(x[:,0])))
        h= h+ self.i_h(x[:,1])
        h= F.relu(self.h_h(h))
        h= h+self.i_h(x[:,2])
        h= F.relu(self.h_h(h))
        return self.h_o(h)

In [40]:
lrnr = Learner(dls, LMModel1(len(vocab),64), loss_func=F.cross_entropy, metrics=accuracy)

In [41]:
lrnr.fit_one_cycle(4,1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.803114,1.888883,0.467079,00:01
1,1.418347,1.746220,0.476824,00:00
2,1.406415,1.674528,0.493701,00:01
3,1.370640,1.663257,0.466128,00:01


In [64]:
x,y = dls.one_batch()

In [65]:
x

tensor([[ 0,  1,  2],
        [ 1,  3,  1],
        [ 4,  1,  5],
        [ 1,  6,  1],
        [ 7,  1,  8],
        [ 1,  9,  1],
        [10,  1, 11],
        [ 1, 12,  1],
        [13,  1, 14],
        [ 1, 15,  1],
        [16,  1, 17],
        [ 1, 18,  1],
        [19,  1, 20],
        [ 1, 20,  0],
        [ 1, 20,  2],
        [ 1, 20,  3],
        [ 1, 20,  4],
        [ 1, 20,  5],
        [ 1, 20,  6],
        [ 1, 20,  7],
        [ 1, 20,  8],
        [ 1, 20,  9],
        [ 1, 21,  1],
        [21,  0,  1],
        [21,  2,  1],
        [21,  3,  1],
        [21,  4,  1],
        [21,  5,  1],
        [21,  6,  1],
        [21,  7,  1],
        [21,  8,  1],
        [21,  9,  1],
        [22,  1, 22],
        [ 0,  1, 22],
        [ 2,  1, 22],
        [ 3,  1, 22],
        [ 4,  1, 22],
        [ 5,  1, 22],
        [ 6,  1, 22],
        [ 7,  1, 22],
        [ 8,  1, 22],
        [ 9,  1, 23],
        [ 1, 23,  0],
        [ 1, 23,  2],
        [ 1, 23,  3],
        [ 

In [71]:
y

tensor([ 1,  4,  1,  7,  1, 10,  1, 13,  1, 16,  1, 19,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1, 21, 21, 21, 21, 21, 21, 21, 21, 21, 22,  0,  2,  3,  4,
         5,  6,  7,  8,  9,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 25,  0,  2,  3])

In [80]:
lrnr.model(x)[0].detach()

tensor([-3.0642,  0.3804, -3.1812, -3.6344, -2.6501, -3.6768, -3.3363, -3.1496,
        -3.4109, -3.3065, -4.9465, -4.9160, -5.0131, -4.6013, -4.5047, -4.0049,
        -4.3536, -4.8198, -4.4614, -4.7675, -3.4891, -3.1594, -3.3581, -2.8839,
        -4.0839, -2.7119, -3.3432, -3.1585,  2.1528,  5.3299])

In [91]:
x[50]

tensor([ 1, 23,  9])

In [95]:
lrnr.predict(x[0])

IndexError: too many indices for tensor of dimension 1

In [88]:
x

tensor([[ 0,  1,  2],
        [ 1,  3,  1],
        [ 4,  1,  5],
        [ 1,  6,  1],
        [ 7,  1,  8],
        [ 1,  9,  1],
        [10,  1, 11],
        [ 1, 12,  1],
        [13,  1, 14],
        [ 1, 15,  1],
        [16,  1, 17],
        [ 1, 18,  1],
        [19,  1, 20],
        [ 1, 20,  0],
        [ 1, 20,  2],
        [ 1, 20,  3],
        [ 1, 20,  4],
        [ 1, 20,  5],
        [ 1, 20,  6],
        [ 1, 20,  7],
        [ 1, 20,  8],
        [ 1, 20,  9],
        [ 1, 21,  1],
        [21,  0,  1],
        [21,  2,  1],
        [21,  3,  1],
        [21,  4,  1],
        [21,  5,  1],
        [21,  6,  1],
        [21,  7,  1],
        [21,  8,  1],
        [21,  9,  1],
        [22,  1, 22],
        [ 0,  1, 22],
        [ 2,  1, 22],
        [ 3,  1, 22],
        [ 4,  1, 22],
        [ 5,  1, 22],
        [ 6,  1, 22],
        [ 7,  1, 22],
        [ 8,  1, 22],
        [ 9,  1, 23],
        [ 1, 23,  0],
        [ 1, 23,  2],
        [ 1, 23,  3],
        [ 